In [0]:
# =============================================================================
# NOTEBOOK  : silver_store_inventory
# PURPOSE   : bronze.store_inventory → silver.store_inventory
# LAYER     : Silver (clean, flatten nested JSON, merge)
# FREQUENCY : Every 15 minutes
# PATTERN   : spark.read + audit watermark (production pattern)
#
# TRANSFORM:
#   - last_restocked_date : String → DateType
#   - expiry_date         : String → DateType (null for non-perishables)
#   - temperature_reading : nested JSON string → flattened into 3 columns:
#                           sensor_id, temperature_celsius, humidity
#                           null for non temperature-sensitive products
#
# MERGE & DEDUP LOGIC (SCD Type 1 — latest snapshot wins):
#   Merge key : store_id + product_id
#   Dedup     : dropDuplicates on store_id + product_id
#               assumption: source never sends two different snapshots
#               for same store-product pair in same 15-min file
#
#   Case 1 : store_id + product_id NOT in silver → INSERT
#   Case 2 : pair IN silver, any value changed   → UPDATE all snapshot cols
#   Case 3 : pair IN silver, no change           → IGNORE
#
# NOTE: SCD Type 1 — silver always reflects CURRENT shelf state
#       temperature_reading null for non temperature-sensitive products
#       expiry_date null for non-perishables — both are valid nulls
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_STORE_INVENTORY_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, get_json_object
)
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_store_inventory"]
TARGET_TABLE = cfg["tables"]["silver_store_inventory"]
PIPELINE     = "silver_store_inventory"

In [0]:
# ── Read + Transform + Merge ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    # ── INCREMENTAL READ ──────────────────────────────────────────────────────
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("ingested_at") > lit(last_run_time))
        )
    else:
        bronze_df = spark.read.table(SOURCE_TABLE)
 
    rows_read = bronze_df.count()
    print(f"[READ] {rows_read} new bronze rows since last run")
 
    if rows_read == 0:
        print(f"[SKIP] No new rows to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0,
                  extra={"rows_inserted": "0", "rows_updated": "0"})
        raise SystemExit(0)
 
    # ── TRANSFORM ─────────────────────────────────────────────────────────────
    df = (
        bronze_df
 
        # 1. Date columns: String → DateType
        .withColumn("last_restocked_date",
            to_date(col("last_restocked_date")))
        .withColumn("expiry_date",
            to_date(col("expiry_date")))
 
        # 2. Flatten temperature_reading nested JSON string
        #    Source: {"sensor_id":"SENSOR_27",
        #             "temperature_celsius":1.62,"humidity":61.64}
        #    null when product is not temperature-sensitive
        #    get_json_object returns null when input is null — safe
        .withColumn("sensor_id",
            get_json_object(col("temperature_reading"), "$.sensor_id"))
        .withColumn("temperature_celsius",
            get_json_object(col("temperature_reading"), "$.temperature_celsius")
            .cast(DoubleType()))
        .withColumn("humidity",
            get_json_object(col("temperature_reading"), "$.humidity")
            .cast(DoubleType()))
 
        # 3. Silver audit columns
        .withColumn("processed_at",    current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
        .withColumn("source_system",   lit("store_inventory_jsonl"))
        # ingested_at carried from bronze automatically via .select()
    )
 
    # ── DEDUP ─────────────────────────────────────────────────────────────────
    # Assumption: source never sends two different snapshots for same
    # store-product pair in same 15-min file — dropDuplicates sufficient
    df = df.dropDuplicates(["store_id", "product_id"])
 
    rows_after_dedup = df.count()
    if rows_read != rows_after_dedup:
        print(f"[DEDUP] dropped {rows_read - rows_after_dedup} duplicate rows")
 
    # Enforce silver schema — drops bronze-only columns
    # (temperature_reading raw string, source_file, ingested_date, etc.)
    df = df.select([f.name for f in SILVER_STORE_INVENTORY_SCHEMA.fields])
 
    # ── MERGE INTO SILVER (SCD Type 1) ────────────────────────────────────────
    # All columns updatable — pure snapshot table
    # Case 2: any value changed → UPDATE all snapshot columns
    # Case 1: new store-product pair → INSERT
    # Case 3: no change → no rule fires → IGNORE
    #
    # NULL comparison note:
    #   t.expiry_date != s.expiry_date evaluates to null when both are null
    #   → no update triggered for null-to-null — correct behaviour
    #   → if null → value or value → null, use OR with IS DISTINCT FROM
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(
            df.alias("s"),
            """
            t.store_id   = s.store_id AND
            t.product_id = s.product_id
            """
        )
        .whenMatchedUpdate(
            condition="""
                t.current_quantity    != s.current_quantity                    OR
                t.last_restocked_date != s.last_restocked_date                 OR
                t.shelf_location      != s.shelf_location                      OR
                (t.expiry_date        IS DISTINCT FROM s.expiry_date)          OR
                (t.sensor_id          IS DISTINCT FROM s.sensor_id)            OR
                (t.temperature_celsius IS DISTINCT FROM s.temperature_celsius)  OR
                (t.humidity           IS DISTINCT FROM s.humidity)
            """,
            set={
                "current_quantity":    "s.current_quantity",
                "last_restocked_date": "s.last_restocked_date",
                "shelf_location":      "s.shelf_location",
                "expiry_date":         "s.expiry_date",
                "sensor_id":           "s.sensor_id",
                "temperature_celsius": "s.temperature_celsius",
                "humidity":            "s.humidity",
                "processed_at":        "current_timestamp()",
                "pipeline_run_id":     f"'{run_id}'"
                # ingested_at NOT updated — preserve original bronze landing time
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )
    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))
 
    print(f"[DONE] inserted={rows_inserted} | updated={rows_updated}")
 
    end_audit(
        spark, PROJECT_ROOT, run_id, "SUCCESS",
        rows_read=rows_read,
        rows_written=rows_inserted + rows_updated,
        extra={
            "rows_inserted": str(rows_inserted),
            "rows_updated":  str(rows_updated)
        }
    )
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── CELL 3: Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)
 
# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())
 
# 3. Nulls in key columns
print("Null key columns:",
    spark.read.table(TARGET_TABLE)
    .filter(col("store_id").isNull() | col("product_id").isNull())
    .count()
)
 
# 4. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)